In [105]:
import tifffile
import pandas as pd
import numpy as np
from skimage.measure import regionprops
import warnings
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.decomposition import PCA
import os
from collections import OrderedDict

In [107]:
roi = tifffile.imread('data/Vsx1_3_D_Kn_Ey_cp_masks.tif')
signal = tifffile.imread('data/Vsx1_3_D_Kn_Ey.tif')

In [108]:
if signal.ndim > 2:
  signal = np.moveaxis(signal, 1, -1)

In [109]:
stats = regionprops(roi, signal)

In [110]:
out = {'label': [], 'centroid_x': [], 'centroid_y': [], 'centroid_z': [], 'area': [], 'aspherity': []}

if signal.ndim == 4:
    for i in range(signal.shape[3]):
        out[f'int_{i}'] = []
else:
    out['int'] = []

for i in stats:
    out['label'].append(i.label)
    out['centroid_x'].append(i.centroid[0])
    out['centroid_y'].append(i.centroid[1])
    out['centroid_z'].append(i.centroid[2])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out['area'].append(i.area_convex)
        
    if signal.ndim == 3:
        out['int'].append(i.intensity_mean)
    else:
        for j in range(signal.shape[3]):
            out[f'int_{j}'].append(i.intensity_mean[j])

    out['aspherity'].append(i.axis_major_length/i.equivalent_diameter_area)

outdf = pd.DataFrame(out)

In [ ]:
outdf.to_csv(f'result/quantification.csv', index=False)
